[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/101EDCsADubKAZk67saQ716lCnYo3ODDz/view?usp=sharing)

# RAG Evaluation – Custom Metrics

This notebook demonstrates how to add custom metrics to RAG evaluations. Custom metrics can use `response`, `question`, `contexts`, and optionally `llm` for LLM-as-judge.

**Objectives**
- Install Floeval and configure credentials
- Load a RAG dataset from a JSON file you provide
- Define a custom metric that checks context usage
- Combine custom metrics with built-in RAG metrics
- Run evaluation and inspect results

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration (OpenAI)

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import evaluation components, the `custom_metric` decorator, and the LLM configuration schema.

In [ ]:
from pathlib import Path

from floeval import DatasetLoader, Evaluation
from floeval.api.metrics.custom import custom_metric
from floeval.config.schemas.io.llm import OpenAIProviderConfig


## 4. Define a Custom Metric (Context Usage)

A custom metric is defined that checks whether the response mentions words from the retrieved contexts. It uses `response` and `contexts`; no API key is required.

In [ ]:
@custom_metric(name="mentions_context", threshold=0.5)
def mentions_context(response: str, contexts: list) -> float:
    """Score 0–1 based on how much the response overlaps with context tokens."""
    if not contexts:
        return 0.0
    joined = " ".join(contexts).lower()
    response_tokens = set(response.lower().split())
    overlap = sum(1 for t in response_tokens if t in joined)
    return min(overlap / 6.0, 1.0)

## 5. Load the RAG Dataset

Minimal shapes: JSON with `samples`, or JSONL one record per line (include `contexts`).

```json
{
  "samples": [
    { "user_input": "...", "llm_response": "...", "contexts": ["..."] }
  ]
}
```

**Example file**  
<a href="../datasets/rag_evaluation/amnesty_qa_eng_v3_context.jsonl" download="amnesty_qa_eng_v3_context.jsonl">amnesty_qa_eng_v3_context.jsonl</a>

Provide the dataset `.jsonl` or `.json` path (or upload in Colab), then load it with `DatasetLoader`.


In [ ]:
try:
    from google.colab import files

    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your dataset .jsonl/.json file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    dataset_path = Path(next(iter(uploaded.keys())))
else:
    dataset_path = (
        Path(input("Enter path to dataset .jsonl/.json file: ").strip().strip('"'))
        .expanduser()
        .resolve()
    )
    if not dataset_path.exists():
        raise FileNotFoundError(f"File not found: {dataset_path}")
    else:
        print("✅ Dataset file found.")


### Resolve Dataset Path

Provide `dataset_path` via file upload in Colab or local `.jsonl`/`.json` path input in Jupyter.


In [ ]:
dataset = DatasetLoader.from_file(dataset_path, partial_dataset=False)
print(f"Dataset loaded from {dataset_path}: {len(dataset.samples)} samples")

### Load RAG Dataset

Load and validate the dataset for combined custom and built-in metric scoring.


## 6. Run Evaluation with Custom + Built-in Metrics

Configure the LLM, build `Evaluation` with `custom:mentions_context` and `faithfulness`, then run. Execute the next three cells in order.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)


### Configure LLM Provider

Build `OpenAIProviderConfig` for built-in metrics that require an LLM provider.


In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["custom:mentions_context", "faithfulness"],
    default_provider="ragas",
)


### Build Evaluation Object

Configure `Evaluation` with both custom and built-in RAG metrics.


In [ ]:
results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)


### Run Evaluation

Execute `evaluation.run()` to compute and print aggregate metric scores.


## Summary

This notebook demonstrated how to add custom metrics to RAG evaluations and combine them with built-in metrics.

The key components included:

1. **Custom Metric Definition**: Defined `mentions_context(response, contexts)` — a custom metric that checks context usage, using `response` and `contexts` (no API key required).
2. **RAG Dataset Loading**: Loaded a RAG dataset from JSON with `user_input`, `llm_response`, and `contexts` for each sample.
3. **LLM Configuration**: The OpenAI-compatible provider was configured for built-in RAG metrics (faithfulness).
4. **Evaluation Execution**: Ran evaluation with both the custom `mentions_context` metric and the built-in `faithfulness` metric.
5. **Results Inspection**: Aggregate scores and per-sample metrics were accessed through `results.aggregate_scores` and `results.sample_results`.

This example showcases the workflow for evaluating RAG with custom metrics that use `response`, `question`, `contexts`, and optionally `llm` for LLM-as-judge.